<a href="https://colab.research.google.com/github/ericjenn/working-groups/blob/ericjenn-srpwg-wg1/safety-related-profile/documents/conv_specification_example/tests/conv_onnx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install onnx onnxruntime

In [2]:
import onnx
from onnx import TensorProto
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info)
from onnx.checker import check_model

In [4]:
# @title Standard max pooling (1 channel)
import numpy as np
from onnx import *
import onnxruntime

# Create inputs
x = helper.make_tensor_value_info('x', TensorProto.DOUBLE, [1, 1, 8, 8])

# Create a node (MaxPool) with input/outputs
node_def = make_node(
    'MaxPool', # node name
    ['x'], # inputs
    ['y', 'Indices'], # outputs
    ceil_mode = 0,
    dilations=[1,1],
    kernel_shape=[3,3],
    pads=[1, 0, 1, 0],
    strides=[1, 1],
    auto_pad='NOTSET',
#    group=1, # Standard convolution
)

# Create the graph
graph_def = make_graph(
    [node_def],
    'test-maxpool',
    [x],
    [helper.make_tensor_value_info('y', TensorProto.DOUBLE, [1, 1, 8, 6]), 
     helper.make_tensor_value_info('Indices', TensorProto.INT64, [1, 1, 8, 6])],
)

onnx_model = make_model(graph_def)

# Let's freeze the opset.
del onnx_model.opset_import[:]
opset = onnx_model.opset_import.add()
opset.domain = ''
opset.version = 15
onnx_model.ir_version = 8

# Verify the model
check_model(onnx_model)

# Print a human-readable representation of the graph
#print(onnx.helper.printable_graph(onnx_model.graph))

# Do inference
sess = onnxruntime.InferenceSession(onnx_model.SerializeToString(),
                        providers=["CPUExecutionProvider"])

# Initialize tensors
#x = numpy.ones((1, 1, 8, 8), dtype=numpy.float32)

x = np.random.normal(3, 2.5, size=(1, 1, 8, 8))

y = sess.run(None, {'x': x})[0]
Indices = sess.run(None, {'x': x})[1]

print("X shape:", x.shape)
print("X:", x)
print("Y shape:", y.shape)
print("Y:", y)
print("Indices shape:", Indices.shape)
print("Indices:", Indices)


X shape: (1, 1, 8, 8)
X: [[[[ 5.86963130e+00  2.71759363e+00  3.78365263e+00  3.00880650e+00
     2.92421210e+00  3.10401095e+00 -2.85957581e+00  2.23624133e+00]
   [ 4.27195341e+00  4.81858677e+00  4.78816825e+00  4.25291443e-02
    -2.60970825e+00  2.69040127e+00  1.18956971e+00  1.26466551e+00]
   [ 1.78011553e+00  2.73377129e+00  1.83015543e+00  8.69613035e+00
     4.37925206e+00  5.07585911e+00  1.23064239e-01  3.16863438e+00]
   [ 3.86076845e+00  4.91977731e+00  2.73732444e+00  4.70577704e+00
     4.02096826e+00  5.80127849e+00  6.54240243e+00  3.27128013e+00]
   [ 4.80388950e+00  2.48609891e+00 -1.58169752e+00  6.42452178e+00
     3.56978414e+00  2.81847655e+00  4.58124995e+00  2.26728895e+00]
   [ 5.42997760e+00  6.28004063e+00  2.49791359e+00  3.28307745e+00
     7.45401849e+00  1.20631370e+00 -1.08235947e+00  2.30303478e+00]
   [-1.77731021e+00 -3.62543516e+00  1.13829782e+00  5.31640420e+00
    -6.33076262e-01  1.48598510e+00  4.76903503e+00 -7.98160972e-03]
   [-2.11966649e